<a href="https://colab.research.google.com/github/smcgovern-proj/neuromatch_ecog_project/blob/main/broadband_mean_electrode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# broadband_mean_electrode

This notebook processes electrophysiological (ECoG) data from a stimulus-presentation experiment (specifically the "Faces vs. Houses" dataset). It extracts high-frequency broadband (gamma) amplitude envelopes, segments them into trial epochs, and computes time-resolved statistics (mean and variance).

---

### Pipeline Configuration

The pipeline uses the following default parameters, designed in accordance with Kai Miller's broadband spectral analysis methods:

* **Sampling Rate ($f_s$):** $1000\text{ Hz}$

* **Frequency Band:** $[70, 150]\text{ Hz}$ (Broadband high-gamma range)


* **Epoch Window:** $-199\text{ ms}$ to $+400\text{ ms}$ relative to stimulus onset ($t_{\text{on}}$)



---

### Core Functions

#### 1. `get_epoch`

Segments the continuous voltage timeseries into discrete trial windows.

* **Inputs:**
* `V`: Raw voltage array of shape `(samples, electrodes)`.


* `t_on`: Stimulus onset timestamps.


* `start_offset` / `end_offset`: Temporal window bounds.




* **Returns:** A 3D numpy array of shape `(n_epochs, epoch_length, n_electrodes)`.



#### 2. `get_envelope`

Extracts the instantaneous amplitude envelope of the signal.

* **Methodology:** Applies a 3rd-order Butterworth bandpass filter ($70\text{--}150\text{ Hz}$) and computes the analytical signal's magnitude using the Hilbert transform.


* **Returns:** Amplitude envelope with the same shape as the input voltage matrix.



#### 3. `extract_envelope_bins`

Slices the calculated trial envelopes into overlapping temporal windows and extracts features.

* **Default Parameters:** Window length of $60$ samples ($60\text{ ms}$) and step size of $40$ samples ($40\text{ ms}$).


* **Metrics Calculated:** Mean amplitude and variance per time bin.


* **Returns:** A nested dictionary containing structured trial, electrode, and window metrics.



#### 4. `results_to_array`

Converts the nested results dictionary back into structured, flat numpy arrays for downstream machine learning or statistical analysis.

* **Returns:** `means` and `variances`, both of shape `(n_epochs, n_windows, n_electrodes)`.

### To-do
1. Separate between faces and houses
2. Need to check if oscillations are true


In [ ]:
# @title Data retrieval
import os, requests

fname = 'faceshouses.npz'
url = "https://osf.io/argh7/download"

if not os.path.isfile(fname):
  try:
    r = requests.get(url)
  except requests.ConnectionError:
    print("!!! Failed to download data !!!")
  else:
    if r.status_code != requests.codes.ok:
      print("!!! Failed to download data !!!")
    else:
      with open(fname, "wb") as fid:
        fid.write(r.content)

In [ ]:
# @title Imports and plot settings

from matplotlib import rcParams
from matplotlib import pyplot as plt
from scipy import signal

rcParams['figure.figsize'] = [20, 4]
rcParams['font.size'] = 15
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['figure.autolayout'] = True

In [ ]:
# @title Data loading
import numpy as np

alldat = np.load(fname, allow_pickle=True)['dat']

In [ ]:
# @title Configure

# Window preset per Kai Miller's paper
start_offset = -199
end_offset = 400

bandpass = [70, 150] #will need to check if oscillations are true
fs = 1000

In [ ]:
# @title Epoch segmentation
def get_epoch(V, t_on, start_offset, end_offset):
  # Epoch
  epochs_list = []

  for on_time in t_on:
      start = on_time + start_offset
      end = on_time + end_offset

      # Window
      trial_window = V[start:end]
      epochs_list.append(trial_window)

  # Convert no np.array
  epochs = np.array(epochs_list)

  return epochs

In [ ]:
# @title Envelope math
def get_envelope(V):
  '''
  Get envelope for voltage using bandpass filter and Hilbert transform

  Args:
    V: float32 np.array (samples, electrodes)

  Return:
    env: amplitude envelope, same shape as V
  '''
  # Design bandpass filter
  b, a = signal.butter(3, bandpass, btype='bandpass', fs=fs)

  # Filter each electrode separately (axis=0 is time/samples)
  filtered_V = signal.filtfilt(b, a, V, axis=0)

  # Get envelope via Hilbert transform
  env = np.abs(signal.hilbert(filtered_V, axis=0))

  return env


def extract_envelope_bins(epochs, win_length, step):
  '''
  Break epochs into overlapping time bins and compute envelope statistics.

  Args:
    epochs: np.array (n_epochs, samples, electrodes)
    win_length: window length in samples
    step: window step in samples (overlap = win_length - step)

  Return:
    results: dict with structure:
      results['trial'][trial_idx]['electrode'][elec_idx] = {
        'mean': np.array (n_windows,),
        'variance': np.array (n_windows,)
      }
  '''
  n_epochs, epoch_length, n_electrodes = epochs.shape

  # Calculate number of windows
  n_windows = int(np.floor((epoch_length - win_length) / step)) + 1

  # Initialize results structure
  results = {
    'trial': [
      {'electrode': [{} for _ in range(n_electrodes)]}
      for _ in range(n_epochs)
    ],
    'metadata': {
      'n_epochs': n_epochs,
      'n_windows': n_windows,
      'n_electrodes': n_electrodes,
      'win_length': win_length,
      'step': step
    }
  }

  # Process each trial
  for trial_idx, epoch in enumerate(epochs):
    # epoch shape: (samples, electrodes)

    # Calculate envelope for entire trial
    envelope = get_envelope(epoch)  # Shape: (samples, electrodes)

    # Process each electrode
    for elec_idx in range(n_electrodes):
      envelope_signal = envelope[:, elec_idx]  # Shape: (samples,)

      mean_amplitudes = []
      variances = []

      # Extract time bins with overlap
      for i in range(n_windows):
        start = i * step
        end = start + win_length

        # Get envelope in this time bin
        bin_envelope = envelope_signal[start:end]

        # Calculate mean amplitude and variance
        mean_amp = np.mean(bin_envelope)
        var = np.var(bin_envelope)

        mean_amplitudes.append(mean_amp)
        variances.append(var)

      # Store results for this electrode
      results['trial'][trial_idx]['electrode'][elec_idx] = {
        'mean': np.array(mean_amplitudes),
        'variance': np.array(variances)
      }

  return results


def results_to_array(results):
  '''
  Convert structured results dict to numpy arrays for easier analysis.

  Return:
    means: np.array (n_epochs, n_windows, n_electrodes)
    variances: np.array (n_epochs, n_windows, n_electrodes)
  '''
  n_epochs = results['metadata']['n_epochs']
  n_windows = results['metadata']['n_windows']
  n_electrodes = results['metadata']['n_electrodes']

  means = np.zeros((n_epochs, n_windows, n_electrodes))
  variances = np.zeros((n_epochs, n_windows, n_electrodes))

  for trial_idx in range(n_epochs):
    for elec_idx in range(n_electrodes):
      means[trial_idx, :, elec_idx] = results['trial'][trial_idx]['electrode'][elec_idx]['mean']
      variances[trial_idx, :, elec_idx] = results['trial'][trial_idx]['electrode'][elec_idx]['variance']

  return means, variances


In [ ]:
# Get data across all subjects

clean_data_variances = []
clean_data_means = []

for subject in alldat:
  dat = subject[0]

  V = dat['V'].astype('float32')
  epochs = get_epoch(V, dat['t_on'], start_offset, end_offset)
  results = extract_envelope_bins(epochs, win_length=60, step=40)
  means, variances = results_to_array(results)

  clean_data_variances.append(means)
  clean_data_means.append(clean_data_means)